<a href="https://colab.research.google.com/github/Serge3leo/temp-cola/blob/main/ruSO/1064719-Как-найти-период-десятичной-дроби-1-n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Как найти период десятичной дроби 1/n

К моему [ответу](TODO) на этот [вопрос ruSO](
https://ru.stackoverflow.com/q/1064719/430734).

## Оглавление

- [Общее описание](#общее-описание)
- [Рецепты из документации Python](#рецепты-из-документации-python)
- [Порядок числа по модулю](#порядок-числа-по-модулю)
- [Период десятичной дроби](#период-десятичной-дроби)
- [Максимально доступная длина периода](#максимально-доступная-длина-периода)
- [Ссылки](#ссылки)

## Общее описание

Структура периодической десятичной дроби для $n, d \in \mathbb{N}$:
$$\frac{n}{d} = \left\lfloor \frac{n}{d} \right\rfloor,e_1 e_2...e_p (q_1 q_2...q_r)$$
, где $e_1 e_2...e_p$ - предпериод и $q_1 q_2...q_r$ - период.

Легко показать, что если $d = 2^x 5^y B$, где $НОД(B, 10) = 1$, то
выполняются следующие соотношения:
$$p = max(x, y)$$
$$10^r \equiv 1 \pmod{B}$$

Т.е. $r$ является [показателем](
https://ru.wikipedia.org/wiki/Порядок_числа_по_модулю) 10 по
модулю $B$:
$$r = P_B(10)$$

Основные полезные свойства показателя $a$ по модулю $m$:
- $P_m(a)$ является делителем значения функции Кармайкла $\lambda(m)$,
  которое, в свою очередь, является делителем значения функции
  Эйлера $\varphi(m)$;
- $P_m(a) = НОК(P_{p_1^{k_1}}(a), P_{p_2^{k_2}}(a), ...)$ для
  разложения $m$ на простые множители $p_1^{k_1} p_2^{k_2}...$.

P.S.

Если $n, d \in \mathbb{Z}$, то
$\left\lfloor \frac{n}{d} \right\rfloor$ следует заменить
на $int(\frac{n}{d})$.

## Рецепты из документации Python

Вероятно, они не очень хорошо будут справляться с факторизацией
очень больших чисел, но примерно до $2^{64}$, работают более менее
приемлемо.

Функция `factor_pk()` является вариантом `factor()` для возврата
списка кортежей `(<простое>, <степень>)`.

In [1]:
import contextlib
import decimal
import itertools
import math
import re
import time

# Источник:
# https://docs.python.org/3/library/itertools.html#itertools-recipes
def iter_index(iterable, value, start=0, stop=None):
    "Return indices where a value occurs in a sequence or iterable."
    # iter_index('AABCADEAF', 'A') → 0 1 4 7
    seq_index = getattr(iterable, 'index', None)
    if seq_index is None:
        iterator = itertools.islice(iterable, start, stop)
        for i, element in enumerate(iterator, start):
            if element is value or element == value:
                yield i
    else:
        stop = len(iterable) if stop is None else stop
        i = start
        with contextlib.suppress(ValueError):
            while True:
                yield (i := seq_index(value, i, stop))
                i += 1
def sieve(n):
    "Primes less than n."
    # sieve(30) → 2 3 5 7 11 13 17 19 23 29
    if n > 2:
        yield 2
    data = bytearray((0, 1)) * (n // 2)
    for p in iter_index(data, 1, start=3, stop=math.isqrt(n) + 1):
        data[p*p : n : p+p] = bytes(len(range(p*p, n, p+p)))
    yield from iter_index(data, 1, start=3)
def factor_pk(n) -> "Iterator[tuple[int, int]]":
    "Разложение n на простые (по p**k)"
    for prime in sieve(math.isqrt(n) + 1):
        if not n % prime:
            n //= prime
            k = 1
            while not n % prime:
                n //= prime
                k += 1
            yield (prime, k)
            if n == 1:
                return
    if n > 1:
        yield (n, 1)

In [2]:
assert list(sieve(30)) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert list(factor_pk(99)) == [(3, 2), (11, 1)]
assert list(factor_pk(1_000_000_000_000_007)) == [(47, 1), (59, 1), (360620266859, 1)]
assert list(factor_pk(1_000_000_000_000_403)) == [(1000000000000403, 1)]
print("Хорь")

Хорь


## Порядок числа по модулю

Для порядка, используем функцию Кармайкла и функции стандартной
библиотеки.

In [3]:
def factor_carmichael_lambda_pk(p, k):
    """[Функция Кармайкла](
    https://ru.wikipedia.org/wiki/Функция_Кармайкла)
    """
    if p > 2:
        yield from factor_pk(p - 1)
        yield (p, k - 1)
    elif k > 2:
        yield (2, k - 2)
    elif k == 2:
        yield (2, 1)
def power_order_pk(p, k, a):
    m = p**k
    lmbd_pk = factor_carmichael_lambda_pk(p, k)
    qs = [1]
    for p1, k1 in lmbd_pk:
        qs = [q * p1**j for j in range(1 + k1) for q in qs]
    qs.sort()
    for q in qs:
        if pow(a, q, m) == 1:
            break
    return q
def power_order(m, a):
    """[Порядок числа по модулю](
    https://ru.wikipedia.org/wiki/Порядок_числа_по_модулю)
    [Multiplicative order](
    https://rosettacode.org/wiki/Multiplicative_order#Python)
    """
    assert math.gcd(m, a) == 1
    orders = []
    for p, k in factor_pk(m):
        orders.append(power_order_pk(p, k, a))
    return math.lcm(*orders)

In [4]:
assert power_order(7, 10) == 6
assert power_order(49, 10) == 42
assert power_order(2**16 - 3, 10) == 14910
assert power_order(2**16 + 1, 10) == 2**16
print("Хорь")

Хорь


## Период десятичной дроби

Зная длины предпериода и периода можно получать цифры периода делением
в столбик не по одной цифре, а группами (по 19 или по 9).  Однако,
в Python тоже самое реализовано в операции
`decimal.Decimal(self.numerator)/self.denominator` модуля `decimal`,
грех не воспользоваться.

In [5]:
class frac_period:
    def __init__(self, denominator=1, numerator=1):
        """Получаем параметры периода дроби `numerator/denominator`
        Популярно: [Периодические дроби](
            https://cyberleninka.ru/article/n/periodicheskie-drobi)
        """
        self.numerator = numerator
        self.denominator = denominator
        self.int_digits = len(str(abs(numerator//denominator)))
        pow2 = 0
        while not denominator%2:
            denominator //= 2
            pow2 += 1
        pow5 = 0
        while not denominator%5:
            denominator //= 5
            pow5 += 1
        self.preperiod_digits = max(pow2, pow5)
        self.reptend_denominator = denominator
        if denominator > 1:
            self.reptend_digits = power_order(denominator, 10)
        else:
            self.reptend_digits = 0
        self.ffrac = None
    def prepare_frac(self):
        if self.ffrac is None:
            self._prepare_frac()
    def _prepare_frac(self):
        with decimal.localcontext(
                prec=self.int_digits + self.preperiod_digits
                     + self.reptend_digits,
                rounding=decimal.ROUND_DOWN) as ctx:
            self.dfrac = decimal.Decimal(self.numerator)/self.denominator
        self.ffrac = f"{self.dfrac:f}"
        dot = self.ffrac.find(".")
        if dot >= 0:
            self.reptend_pos = dot + 1 + self.preperiod_digits
        else:
            self.reptend_pos = len(self.ffrac)
    def reptend_str(self):
        self.prepare_frac()
        return self.ffrac[self.reptend_pos:
                          self.reptend_pos + self.reptend_digits]
    def __repr__(self):
        self.prepare_frac()
        if not self.reptend_digits:
            return self.ffrac[:self.reptend_pos]
        return (self.ffrac[:self.reptend_pos] + "("
                + self.ffrac[self.reptend_pos:
                             self.reptend_pos + self.reptend_digits]
                + ")")

In [6]:
assert frac_period(7).reptend_digits == 6
assert frac_period(7).reptend_str() == "142857"
assert frac_period(35).preperiod_digits == 1
assert frac_period(35).reptend_str() == "285714"
assert str(frac_period(3)) == "0.(3)"
assert str(frac_period(5)) == "0.2"
assert str(frac_period(60)) == "0.01(6)"
assert str(frac_period(350)) == "0.00(285714)"
assert str(frac_period(19)) == "0.(052631578947368421)"
assert str(frac_period(numerator=9999943,
                       denominator=40*17)) == "14705.798(5294117647058823)"
assert re.match(r"0.\(0*811622.*837919\)", str(frac_period(12321)))
assert re.match(r"0.\(0*152585.*526527\)", str(frac_period(2**16 + 1)))
assert re.match(r"0.\(0*100000570.*840877193\)", str(frac_period(9999943)))
print("Хорь")

Хорь


## Максимально доступная длина периода

Похоже, основным ограничением являетется цена операций
`(decimal.Decimal(self.numerator)/self.denominator)`
и `f"{self.dfrac:f}"`.  Которая растёт примерно линейно
от `len(f.reptend_str())` до тех пор, пока памяти достаточно.

In [7]:
for n in [2**16 + 1, 9999943, 2**31-1, 2**31 + 11,
          # 7*2**29-1, 5*2**32-1
          ]:
    start_time = time.time()
    f = frac_period(n)
    ctr_time = time.time()
    print(f"{n=}: конструктор: {ctr_time - start_time:.4f} секунд")
    r = f.reptend_str()
    str_time = time.time()
    print(f"{len(r)=}: строка периода: {str_time - ctr_time:.4f} секунд")
print("Хорь")

n=65537: конструктор: 0.0001 секунд
len(r)=65536: строка периода: 0.0006 секунд
n=9999943: конструктор: 0.0003 секунд
len(r)=9999942: строка периода: 0.0567 секунд
n=2147483647: конструктор: 0.0030 секунд
len(r)=195225786: строка периода: 1.3780 секунд
n=2147483659: конструктор: 0.0252 секунд
len(r)=715827886: строка периода: 5.4494 секунд
Хорь


# Ссылки

- [Периодические дроби, Храбров А.И.](
  https://cyberleninka.ru/article/n/periodicheskie-drobi)
- [Порядок числа по модулю, Википедия](
  https://ru.wikipedia.org/wiki/Порядок_числа_по_модулю)
- [Функция Кармайкла, Википедия](
  https://ru.wikipedia.org/wiki/Функция_Кармайкла)
- [Itertools Recipes, Python](
  https://docs.python.org/3/library/itertools.html#itertools-recipes)
- [Multiplicative order, Rosetta Code](
  https://rosettacode.org/wiki/Multiplicative_order#Python)